# Exploitation de données électorales avec Python

Antoine Rustenholz, Aziz Seghaier, Jasmin Neveu
31.04.2026

In [1]:
import pandas as pd
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)

/tmp/ipykernel_29127/572529936.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## 1. Explorations générales

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 1
</div>
</div>
<div class="callout-body-container callout-body">

Créer ou mettre à jour les variables suivantes:

-   `code_commune`: En utilisant la variable déjà existante et le département, remplacer la valeur `code_commune` pour constituer un vrai code commune. Par exemple, pour Montrouge, vous devriez obtenir 92049.

-   `candidat`: créer une colonne avec le prenom et le nom mis ensemble, en n’oubliant pas de mettre un espace. Ne pas éliminer les bulletins abstentions, blancs ou nuls, nous allons les exploiter ultérieurement.

</div>
</div>

In [ ]:
# Pour créer un vrai code commune, il suffit de concaténer les codes du départements et de la commune
# en ajoutant des zéros afin d'avoir toujours un code à 5 chiffres.
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2) +
    df['code_commune'].astype(str).str.zfill(3)
)

# On fait juste attentionau valeurs manquantes dans la colonne prenom pour les votes non exprimés.
df['candidat'] = (
    df['prenom'].fillna('') + ' ' + df['nom']
)

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 2
</div>
</div>
<div class="callout-body-container callout-body">

Compléter la phrase suivante grâce à Python:
En 2022, il y avait XXXXX candidats à l’élection présidentielle.
*Note: Attention aux votes non exprimés et aux abstentions*

</div>
</div>

In [ ]:
# Il suffit de compter le nombre de valeurs différentes pour candidats en supprimant les abstentions, les votes blancs et nuls.
f"En 2022, il y avait {df['candidat'].nunique()-3} candidats à l'élection présidentielle."

"En 2022, il y avait 12 candidats à l'élection présidentielle."

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 3
</div>
</div>
<div class="callout-body-container callout-body">

Calculer les scores nationaux de chaque candidat. Représenter dans ce tableau, pour chaque candidat, le nombre de voix et le pourcatage des votes exprimés (c’est-à-dire en retirant abstentions et votes non exprimés).
Représenter cela dans un *dataframe* ou, pour avoir tous les points, dans un tableau mis en forme via `great_tables` (il n’est pas obligatoire d’aller aussi loin dans la mise en forme mais essayez d’obtenir un beau tableau tout de même).
*Note: vous pouvez contrôler vos résultats obtenus avec cette page.*

</div>
</div>

In [ ]:
# Création d'une table auxiliaire
resultats = (
    df[['candidat', 'voix']]
    .groupby('candidat', as_index = False).sum()
)

# Suppression des non-candidats
resultats = resultats[~resultats['candidat'].isin([' abstentions', ' nuls', ' blancs'])]

# Calcul du nombre total de voix exprimées
nombre_total = resultats['voix'].sum()

# Création de la variable score
resultats['score'] = resultats['voix'] / nombre_total

# Tri
resultats = resultats.sort_values('voix', ascending=False)


# Génération du tableau
from great_tables import GT

(
    GT(resultats)
    .tab_header(
        title = "Elections présidentielles 2022",
        subtitle = "Résultats du premier tour (10 avril 2022)"
    )
    .cols_label( # Renommage des colonnes
        candidat = "Candidat",
        voix = "Nombre votes (total)",
        score = "Score (% votes exprimés)"
    )
    .fmt_number( # Formatage des nombres de voix
        columns = "voix",
        decimals = 0,
        sep_mark = ' '
    )
    .fmt_percent( # Formatage du score en pourcentage
        columns = "score",
        decimals = 2
    )
)

GT(_tbl_data=                 candidat     voix     score
4         Emmanuel MACRON  9783058  0.278458
8           Marine LE PEN  8133828  0.231516
7      Jean-Luc MÉLENCHON  7712520  0.219524
14           Éric ZEMMOUR  2485226  0.070738
12       Valérie PÉCRESSE  1679001  0.047790
13          Yannick JADOT  1627853  0.046334
6           Jean LASSALLE  1101387  0.031349
5          Fabien ROUSSEL   802422  0.022840
10  Nicolas DUPONT-AIGNAN   725176  0.020641
3            Anne HIDALGO   616478  0.017547
11        Philippe POUTOU   268904  0.007654
9        Nathalie ARTHAUD   197094  0.005610, _body=<great_tables._gt_data.Body object at 0x7f90f768eb10>, _boxhead=Boxhead([ColInfo(var='candidat', type=<ColInfoTypeEnum.default: 1>, column_label='Candidat', column_align='left', column_width=None), ColInfo(var='voix', type=<ColInfoTypeEnum.default: 1>, column_label='Nombre votes (total)', column_align='right', column_width=None), ColInfo(var='score', type=<ColInfoTypeEnum.default: 1>, column_label='Score (% votes exprimés)', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f90f751c690>, _spanners=Spanners([]), _heading=Heading(title='Elections présidentielles 2022', subtitle='Résultats du premier tour (10 avril 2022)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f90f751cb90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f90f768ec40>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f90f751ccd0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7f90f751cf50>, <great_tables._gt_data.FormatInfo object at 0x7f90f768eea0>], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_

## 2. Comparaison des scores départements aux moyennes nationales.

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 4
</div>
</div>
<div class="callout-body-container callout-body">

Créer un *dataframe* nommé `score_departements` stockant, pour chaque département, le nombre de vote obtenu pour chaque candidat et le score (en %).

</div>
</div>

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 5
</div>
</div>
<div class="callout-body-container callout-body">

Refaire le lien avec le niveau national pour comparer le score départemental avec le score national. Nommer ce *dataframe* `score_departements`, nous allons le réutiliser par la suite.

</div>
</div>

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 6
</div>
</div>
<div class="callout-body-container callout-body">

Créer une variable `surrepresentation` qui compare, en relatif, les scores nationaux et départementaux.
Par exemple, si un candidat a un score de 30% dans un département mais de 15% ailleurs, la valeur de `surrepresentation` sera égale à 100 (%).


</div>
</div>

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 7
</div>
</div>
<div class="callout-body-container callout-body">

Créer une fonction pour représenter une figure similaire à Figure 1 pour un candidat donné des
principales surreprésentations (en valeur absolue) par département.

</div>
</div>

## 3. Un peu de cartographie

<div class="callout callout-style-default callout-tip callout-titled">
<div class="callout-header d-flex align-content-center">
<div class="callout-icon-container">
<i class="callout-icon"></i>
</div>
<div class="callout-title-container flex-fill">
Question 8
</div>
</div>
<div class="callout-body-container callout-body">

Faire une fonction permettant de restreindre `score_departements` en fonction d’un candidat. Commencer par tester sur Marine Le Pen (créer un nouvel objet, ne pas écraser `score_departements` nous allons l’utiliser à nouveau !).
Faire une jointure au fond de carte des départements et effectuer une carte de la représentation.

</div>
</div>